# V14 Ablation Study

## Design

| ID | Model | What's removed | Tests |
|---|---|---|---|
| **A** | Full V14 | Nothing | Baseline |
| **B** | No Wilson line | wilson_proj → fixed decay=0.5, θ=0 | Content-dependent gauge transport |
| **C** | No Langevin | Settling loop removed, delta = ctx only | Iterative Hopfield settling |
| **D** | No sparsity | Proximal threshold removed | Spectral parsimony |
| **E** | SSM+MLP only | Fiber+settler → real EMA + MLP (param-matched) | **The bar to clear** |

## The Computation Graph (annotated with ablation targets)

```
constellation ─────────────────────────────────────────────┐ (residual)
     │                                                      │
  MagPhaseNorm                                              │
     │                                                      │
     ├──────────────────────────┐                           │
     │                          │                           │
  WilsonFiber              (normed constellation)           │
     │                          │                           │
     │ ┌─ wilson_proj(flat) ──┐ │                           │
     │ │  Linear→SiLU→Linear  │ │  ← ABLATION B removes    │
     │ │  → (decay_t, θ_t)   │ │                           │
     │ └──────────────────────┘ │                           │
     │        │                 │                           │
     │   z_t = decay·exp(iθ)   │                           │
     │        │                 │                           │
     │   complex parallel scan  │                           │
     │   h[t] = z_t·h[t-1]+c_t │                           │
     │        │                 │                           │
     │   Parseval: Re(h·conj(c))│                           │
     │        │                 │                           │
     │     messages             │                           │
     │        │                 │                           │
  LangevinSettler ──────────────┘                           │
     │                                                      │
     │ ┌─ PATH A: ctx = msg_proj(messages) ─── LINEAR ─┐   │
     │ │                                                │   │
     │ ├─ PATH B: Langevin loop ──────────────────────┐ │   │
     │ │  q = normalize(x + ctx)                      │ │   │
     │ │  attractor = softmax(β·q@M_norm^T) @ M       │ │   │
     │ │  x = x + η·(attractor - x) + noise           │ │   │
     │ │  x_mag = threshold(x_mag) ← ABLATION D       │ │   │
     │ └──── ABLATION C removes entire loop ──────────┘ │   │
     │                                                  │   │
     │  delta = (x - x₀) + ctx                         │   │
     │                                                      │
     │  d = per_mode_gate · delta                           │
     └──► constellation + d ◄───────────────────────────────┘
```

## Verdict Table

| Result | Interpretation |
|---|---|
| A > E | Geometry is earning its keep |
| A ≈ E | Geometry is still decorative (V12.2 repeats) |
| A > C ≈ E | Langevin is decorative, Wilson fiber does the work (complex Mamba) |
| A > B ≈ E | Wilson line is the key mechanism |
| A ≈ D | Sparsity doesn't matter yet |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
import math
import time
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join(os.getcwd(), "tiny_shakespeare.txt")
if not os.path.exists(DATA_PATH):
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH) as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Tiny Shakespeare: {len(data):,} chars, vocab {vocab_size}")

In [ ]:
@dataclass
class AblationConfig:
    # Spectral structure
    n_modes: int = 136
    n_subbundles: int = 8
    fiber_dim: int = 256

    # Wilson fiber
    wilson_hidden: int = 192

    # Langevin settler
    n_memory_atoms: int = 256
    n_langevin_steps: int = 2
    beta_min: float = 0.5
    beta_max: float = 5.0
    langevin_eta: float = 0.3
    sparsity_threshold: float = 0.05

    # SSM+MLP ablation
    ssm_mlp_hidden: int = 311     # param-matched to V14 block

    # Model
    n_blocks: int = 8
    vocab_size: int = 65
    max_seq_len: int = 512
    dropout: float = 0.1

    # Training
    learning_rate: float = 1e-3
    min_lr: float = 1e-4
    warmup_steps: int = 750
    lr_hold_steps: int = 1250
    batch_size: int = 16
    seq_len: int = 512
    max_steps: int = 5000         # shorter for ablation (adjust if needed)
    eval_interval: int = 250
    eval_steps: int = 10

    @property
    def subbundle_dim(self):
        return self.fiber_dim // self.n_subbundles

    @property
    def spectral_half_dim(self):
        return self.subbundle_dim // 2 + 1

cfg = AblationConfig(vocab_size=vocab_size)
print(f"Ablation config: {cfg.max_steps} steps, eval every {cfg.eval_interval}")

def get_batch(split_data, c):
    ix = torch.randint(0, len(split_data) - c.seq_len - 1, (c.batch_size,))
    return torch.stack([split_data[i:i+c.seq_len] for i in ix]).to(device)

## Shared Components

In [ ]:
class Constellation:
    def __init__(self, mag, phase):
        self.mag = mag
        self.phase = phase

    def to_complex(self):
        return self.mag * torch.exp(1j * self.phase)

    def to_flat(self):
        return torch.cat([self.mag, self.phase], dim=-1)


class MagPhaseNorm(nn.Module):
    def __init__(self, n_modes):
        super().__init__()
        self.mag_scale = nn.Parameter(torch.ones(n_modes))

    def forward(self, c):
        mag_rms = (c.mag ** 2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-8)
        return Constellation(c.mag / mag_rms * self.mag_scale, c.phase)


class ConstellationEmbedding(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mag_emb = nn.Embedding(cfg.vocab_size, cfg.n_modes)
        self.phase_emb = nn.Embedding(cfg.vocab_size, cfg.n_modes)
        nn.init.uniform_(self.phase_emb.weight, -math.pi, math.pi)
        freqs = torch.zeros(cfg.n_modes)
        for k in range(cfg.n_subbundles):
            off = k * cfg.spectral_half_dim
            freqs[off:off+cfg.spectral_half_dim] = (
                2 * math.pi * torch.fft.rfftfreq(cfg.subbundle_dim, d=1.0))
        self.register_buffer('freqs', freqs)

    def forward(self, token_ids):
        B, T = token_ids.shape
        mag = self.mag_emb(token_ids)
        phase = self.phase_emb(token_ids)
        pos = torch.arange(T, device=token_ids.device).float()
        phase = phase + (pos.unsqueeze(-1) * self.freqs).unsqueeze(0)
        return Constellation(mag, phase)


class ConstellationDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.head = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )

    def forward(self, constellation):
        spectral = constellation.to_complex()
        shd = self.cfg.spectral_half_dim
        subs = spectral.reshape(*spectral.shape[:-1], self.cfg.n_subbundles, shd)
        spatial = torch.fft.irfft(subs, n=self.cfg.subbundle_dim, dim=-1)
        spatial = spatial.reshape(*spectral.shape[:-1], self.cfg.fiber_dim)
        return self.head(self.norm(spatial))


def parallel_scan(alpha, x):
    N, T = alpha.shape
    a, b = alpha, x
    for d in range(int(math.ceil(math.log2(T)))):
        step = 2 ** d
        if step >= T:
            break
        b = torch.cat([b[:, :step],
                        a[:, step:] * b[:, :-step] + b[:, step:]], dim=1)
        a = torch.cat([a[:, :step],
                        a[:, step:] * a[:, :-step]], dim=1)
    return b


def complex_parallel_scan(a_re, a_im, b_re, b_im):
    N, T = a_re.shape
    for d in range(int(math.ceil(math.log2(T)))):
        step = 2 ** d
        if step >= T:
            break
        ar, ai = a_re[:, step:], a_im[:, step:]
        al, ail = a_re[:, :-step], a_im[:, :-step]
        bl, bil = b_re[:, :-step], b_im[:, :-step]
        ab_re = ar * bl - ai * bil
        ab_im = ar * bil + ai * bl
        aa_re = ar * al - ai * ail
        aa_im = ar * ail + ai * al
        b_re = torch.cat([b_re[:, :step], ab_re + b_re[:, step:]], dim=1)
        b_im = torch.cat([b_im[:, :step], ab_im + b_im[:, step:]], dim=1)
        a_re = torch.cat([a_re[:, :step], aa_re], dim=1)
        a_im = torch.cat([a_im[:, :step], aa_im], dim=1)
    return b_re, b_im


def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("Shared components loaded.")

## Model A: Full V14 (Baseline)

All geometric mechanisms active: Wilson fiber + Langevin settling + proximal sparsity.

In [ ]:
class WilsonFiber(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        M = cfg.n_modes
        self.base_decay = nn.Parameter(torch.zeros(M))
        self.wilson_proj = nn.Sequential(
            nn.Linear(2 * M, cfg.wilson_hidden),
            nn.SiLU(),
            nn.Linear(cfg.wilson_hidden, 2 * M),
        )
        nn.init.zeros_(self.wilson_proj[-1].weight)
        nn.init.zeros_(self.wilson_proj[-1].bias)

    def forward(self, constellation):
        B, T, M = constellation.mag.shape
        flat = constellation.to_flat()
        wilson = self.wilson_proj(flat)
        decay_delta = wilson[..., :M]
        phase_rot = wilson[..., M:]
        decay = torch.sigmoid(self.base_decay + decay_delta).clamp(0.01, 0.99)
        theta = torch.tanh(phase_rot) * math.pi
        z_re = decay * torch.cos(theta)
        z_im = decay * torch.sin(theta)
        c_re = constellation.mag * torch.cos(constellation.phase)
        c_im = constellation.mag * torch.sin(constellation.phase)
        z_re_f = z_re.permute(0, 2, 1).reshape(B * M, T)
        z_im_f = z_im.permute(0, 2, 1).reshape(B * M, T)
        c_re_f = c_re.permute(0, 2, 1).reshape(B * M, T)
        c_im_f = c_im.permute(0, 2, 1).reshape(B * M, T)
        h_re_f, h_im_f = complex_parallel_scan(z_re_f, z_im_f, c_re_f, c_im_f)
        h_re_f = F.pad(h_re_f[:, :-1], (1, 0))
        h_im_f = F.pad(h_im_f[:, :-1], (1, 0))
        h_re = h_re_f.reshape(B, M, T).permute(0, 2, 1)
        h_im = h_im_f.reshape(B, M, T).permute(0, 2, 1)
        messages = h_re * c_re + h_im * c_im
        return messages


class LangevinSettler(nn.Module):
    def __init__(self, cfg, use_sparsity=True):
        super().__init__()
        M = cfg.n_modes
        self.memory = nn.Parameter(torch.randn(cfg.n_memory_atoms, 2 * M) * 0.02)
        self.msg_proj = nn.Linear(M, 2 * M, bias=False)
        nn.init.normal_(self.msg_proj.weight, std=0.01)
        self.gate = nn.Parameter(torch.full((M,), -2.0))
        self.K = cfg.n_langevin_steps
        self.eta = cfg.langevin_eta
        self.beta_min = cfg.beta_min
        self.beta_max = cfg.beta_max
        self.threshold = cfg.sparsity_threshold if use_sparsity else 0.0
        self.use_sparsity = use_sparsity

    def forward(self, constellation, messages):
        M = constellation.mag.shape[-1]
        x = constellation.to_flat()
        x0 = x
        ctx = self.msg_proj(messages)
        m_norm = F.normalize(self.memory, dim=-1)
        for k in range(self.K):
            beta = self.beta_min + (self.beta_max - self.beta_min) * k / max(1, self.K - 1)
            q = F.normalize(x + ctx, dim=-1)
            scores = beta * (q @ m_norm.T)
            weights = F.softmax(scores, dim=-1)
            attractor = weights @ self.memory
            grad = attractor - x
            noise = 0.0
            if self.training:
                noise = math.sqrt(2 * self.eta / beta) * torch.randn_like(x)
            x = x + self.eta * grad + noise
            if k == self.K - 1 and self.use_sparsity:
                mag = x[..., :M]
                mag = torch.sign(mag) * F.relu(mag.abs() - self.threshold)
                phase = torch.remainder(x[..., M:] + math.pi, 2 * math.pi) - math.pi
                x = torch.cat([mag, phase], dim=-1)
        delta = (x - x0) + ctx
        g = torch.sigmoid(self.gate)
        return g * delta[..., :M], g * delta[..., M:]


class BlockA(nn.Module):
    """Full V14 block."""
    def __init__(self, cfg):
        super().__init__()
        self.norm = MagPhaseNorm(cfg.n_modes)
        self.fiber = WilsonFiber(cfg)
        self.settler = LangevinSettler(cfg, use_sparsity=True)

    def forward(self, constellation):
        normed = self.norm(constellation)
        messages = self.fiber(normed)
        d_mag, d_phase = self.settler(normed, messages)
        return Constellation(constellation.mag + d_mag, constellation.phase + d_phase)

print(f"Model A (Full V14) block params: {count_params(BlockA(cfg)):,}")

## Model B: No Wilson Line

Replace content-dependent complex recurrence with V13's fixed real decay.
No wilson_proj MLP, no phase rotation. Tests: **does gauge transport matter?**

In [ ]:
class FixedDecayFiber(nn.Module):
    """V13-style fiber: fixed real decay, no content dependence, no phase rotation."""
    def __init__(self, cfg):
        super().__init__()
        self.decay = nn.Parameter(torch.zeros(cfg.n_modes))

    def forward(self, constellation):
        B, T, M = constellation.mag.shape
        c_re = constellation.mag * torch.cos(constellation.phase)
        c_im = constellation.mag * torch.sin(constellation.phase)
        alpha = torch.sigmoid(self.decay).clamp(0.01, 0.99)
        alpha_flat = alpha.unsqueeze(0).expand(B, -1).reshape(B * M, 1).expand(-1, T)
        re_flat = c_re.permute(0, 2, 1).reshape(B * M, T)
        im_flat = c_im.permute(0, 2, 1).reshape(B * M, T)
        h_flat = parallel_scan(
            torch.cat([alpha_flat, alpha_flat], dim=0),
            torch.cat([re_flat, im_flat], dim=0))
        h_flat = F.pad(h_flat[:, :-1], (1, 0))
        h_re = h_flat[:B*M].reshape(B, M, T).permute(0, 2, 1)
        h_im = h_flat[B*M:].reshape(B, M, T).permute(0, 2, 1)
        return h_re * c_re + h_im * c_im


class BlockB(nn.Module):
    """No Wilson line: fixed decay fiber + full Langevin settler."""
    def __init__(self, cfg):
        super().__init__()
        self.norm = MagPhaseNorm(cfg.n_modes)
        self.fiber = FixedDecayFiber(cfg)
        self.settler = LangevinSettler(cfg, use_sparsity=True)

    def forward(self, constellation):
        normed = self.norm(constellation)
        messages = self.fiber(normed)
        d_mag, d_phase = self.settler(normed, messages)
        return Constellation(constellation.mag + d_mag, constellation.phase + d_phase)

print(f"Model B (No Wilson) block params: {count_params(BlockB(cfg)):,}")

## Model C: No Langevin

Remove settling loop entirely. `delta = ctx` only (linear projection of fiber messages).
No memory bank, no Hopfield, no proximal sparsity.
Tests: **is V14 just a complex SSM + linear readout?**

In [ ]:
class CtxOnlySettler(nn.Module):
    """No Langevin loop. Just linear message projection → gated residual."""
    def __init__(self, cfg):
        super().__init__()
        M = cfg.n_modes
        self.msg_proj = nn.Linear(M, 2 * M, bias=False)
        nn.init.normal_(self.msg_proj.weight, std=0.01)
        self.gate = nn.Parameter(torch.full((M,), -2.0))

    def forward(self, constellation, messages):
        M = constellation.mag.shape[-1]
        ctx = self.msg_proj(messages)
        g = torch.sigmoid(self.gate)
        return g * ctx[..., :M], g * ctx[..., M:]


class BlockC(nn.Module):
    """Wilson fiber + linear readout (no Langevin)."""
    def __init__(self, cfg):
        super().__init__()
        self.norm = MagPhaseNorm(cfg.n_modes)
        self.fiber = WilsonFiber(cfg)
        self.settler = CtxOnlySettler(cfg)

    def forward(self, constellation):
        normed = self.norm(constellation)
        messages = self.fiber(normed)
        d_mag, d_phase = self.settler(normed, messages)
        return Constellation(constellation.mag + d_mag, constellation.phase + d_phase)

print(f"Model C (No Langevin) block params: {count_params(BlockC(cfg)):,}")

## Model D: No Sparsity

Full V14 but proximal threshold removed. Langevin settles without enforcing spectral parsimony.
Tests: **does the "few dots on Fourier space" constraint matter?**

In [ ]:
class BlockD(nn.Module):
    """Full V14 block, but use_sparsity=False."""
    def __init__(self, cfg):
        super().__init__()
        self.norm = MagPhaseNorm(cfg.n_modes)
        self.fiber = WilsonFiber(cfg)
        self.settler = LangevinSettler(cfg, use_sparsity=False)

    def forward(self, constellation):
        normed = self.norm(constellation)
        messages = self.fiber(normed)
        d_mag, d_phase = self.settler(normed, messages)
        return Constellation(constellation.mag + d_mag, constellation.phase + d_phase)

print(f"Model D (No Sparsity) block params: {count_params(BlockD(cfg)):,}")

## Model E: SSM+MLP Only (The Bar to Clear)

Strip ALL geometric machinery. Replace with:
- Real-valued EMA (fixed learned decay, like V13)
- Standard MLP (Linear→SiLU→Dropout→Linear)
- Parameter-matched to V14 block size

**If A ≈ E, the geometry is decorative regardless of how elegant the math is.**

In [ ]:
class BlockE(nn.Module):
    """SSM + MLP. No complex numbers, no Parseval, no Langevin, no sparsity."""
    def __init__(self, cfg):
        super().__init__()
        M = cfg.n_modes
        self.norm = MagPhaseNorm(M)
        self.decay = nn.Parameter(torch.zeros(M))
        self.mlp = nn.Sequential(
            nn.Linear(3 * M, cfg.ssm_mlp_hidden),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.ssm_mlp_hidden, 2 * M),
        )
        self.gate = nn.Parameter(torch.tensor(-2.0))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, constellation):
        normed = self.norm(constellation)
        B, T, M = normed.mag.shape
        # Real EMA (V13 fiber)
        c_re = normed.mag * torch.cos(normed.phase)
        c_im = normed.mag * torch.sin(normed.phase)
        alpha = torch.sigmoid(self.decay).clamp(0.01, 0.99)
        alpha_flat = alpha.unsqueeze(0).expand(B, -1).reshape(B * M, 1).expand(-1, T)
        re_flat = c_re.permute(0, 2, 1).reshape(B * M, T)
        im_flat = c_im.permute(0, 2, 1).reshape(B * M, T)
        h_flat = parallel_scan(
            torch.cat([alpha_flat, alpha_flat], dim=0),
            torch.cat([re_flat, im_flat], dim=0))
        h_flat = F.pad(h_flat[:, :-1], (1, 0))
        h_re = h_flat[:B*M].reshape(B, M, T).permute(0, 2, 1)
        h_im = h_flat[B*M:].reshape(B, M, T).permute(0, 2, 1)
        messages = h_re * c_re + h_im * c_im
        # MLP update (V13-style)
        combined = torch.cat([normed.mag, normed.phase, messages], dim=-1)
        delta = self.mlp(combined)
        g = torch.sigmoid(self.gate)
        d_mag = g * delta[..., :M]
        d_phase = g * delta[..., M:]
        return Constellation(constellation.mag + d_mag, constellation.phase + d_phase)

print(f"Model E (SSM+MLP) block params: {count_params(BlockE(cfg)):,}")

In [ ]:
class GPTNano(nn.Module):
    def __init__(self, vocab_size=65, n_embd=128, n_head=4, n_layer=12,
                 block_size=512, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList()
        for _ in range(n_layer):
            self.blocks.append(nn.ModuleDict({
                'ln1': nn.LayerNorm(n_embd),
                'attn_qkv': nn.Linear(n_embd, 3 * n_embd),
                'attn_proj': nn.Linear(n_embd, n_embd),
                'ln2': nn.LayerNorm(n_embd),
                'mlp_fc1': nn.Linear(n_embd, 4 * n_embd),
                'mlp_fc2': nn.Linear(4 * n_embd, n_embd),
            }))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer('causal_mask',
            torch.tril(torch.ones(block_size, block_size)).view(1,1,block_size,block_size))

    def forward(self, idx):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        hd = self.n_embd // self.n_head
        for blk in self.blocks:
            h = blk['ln1'](x)
            qkv = blk['attn_qkv'](h).reshape(B, T, 3, self.n_head, hd)
            q, k, v = qkv.unbind(2)
            q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
            att = (q @ k.transpose(-2,-1)) * (hd**-0.5)
            att = att.masked_fill(self.causal_mask[:,:,:T,:T]==0, float('-inf'))
            y = (F.softmax(att, dim=-1) @ v).transpose(1,2).reshape(B, T, self.n_embd)
            x = x + blk['attn_proj'](y)
            x = x + blk['mlp_fc2'](F.gelu(blk['mlp_fc1'](blk['ln2'](x))))
        return self.lm_head(self.ln_f(x))[:, :-1, :], {}


class AblationModel(nn.Module):
    def __init__(self, cfg, block_class):
        super().__init__()
        self.cfg = cfg
        self.embedding = ConstellationEmbedding(cfg)
        self.blocks = nn.ModuleList([block_class(cfg) for _ in range(cfg.n_blocks)])
        self.decoder = ConstellationDecoder(cfg)

    def forward(self, token_ids):
        c = self.embedding(token_ids)
        for block in self.blocks:
            c = block(c)
        logits = self.decoder(c)[:, :-1, :]
        sp = (c.mag.abs() < 0.01).float().mean().item()
        return logits, {'spectral_sparsity': sp, 'mag_mean': c.mag.abs().mean()}


# Instantiate all models
models = {}
for name, block_cls in [('A: Full V14', BlockA), ('B: No Wilson', BlockB),
                         ('C: No Langevin', BlockC), ('D: No Sparsity', BlockD),
                         ('E: SSM+MLP', BlockE)]:
    models[name] = AblationModel(cfg, block_cls).to(device)

models['GPT-Nano'] = GPTNano(vocab_size=vocab_size, block_size=cfg.seq_len).to(device)

print(f"{'Model':<20} {'Params':>10}")
print('=' * 32)
for name, m in models.items():
    print(f"{name:<20} {count_params(m):>10,}")

## Training

In [ ]:
@torch.no_grad()
def estimate_loss(model, c, is_gpt=False):
    model.eval()
    results = {}
    for name, sd in [('train', train_data), ('val', val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        for _ in range(c.eval_steps):
            b = get_batch(sd, c)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, c.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            if not is_gpt:
                tot_sp += info.get('spectral_sparsity', 0.0)
        n = c.eval_steps
        results[name] = {
            'ce': tot_ce/n, 'acc': tot_ok/tot_n,
            'sparsity': tot_sp/n if not is_gpt else 0.0}
    model.train()
    return results


def train_model(model, c, label='model', is_gpt=False):
    opt = torch.optim.AdamW(model.parameters(), lr=c.learning_rate, weight_decay=0.05)
    mr = c.min_lr / c.learning_rate
    he = c.warmup_steps + c.lr_hold_steps

    def lr_fn(s):
        if s < c.warmup_steps: return s / max(1, c.warmup_steps)
        if s < he: return 1.0
        p = (s - he) / max(1, c.max_steps - he)
        return max(mr, 0.5 * (1.0 + math.cos(math.pi * p)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    hist = {'step':[], 'train_ce':[], 'val_ce':[], 'train_acc':[], 'val_acc':[],
            'train_bpc':[], 'val_bpc':[], 'sparsity':[], 'step_times':[],
            'per_step_loss':[]}

    model.train()
    t0 = time.time()
    np_ = count_params(model)
    print(f"\nTraining {label}: {np_:,} params, {c.max_steps} steps")

    for step in range(c.max_steps + 1):
        if step % c.eval_interval == 0:
            r = estimate_loss(model, c, is_gpt=is_gpt)
            tr, vl = r['train'], r['val']
            hist['step'].append(step)
            hist['train_ce'].append(tr['ce']); hist['val_ce'].append(vl['ce'])
            hist['train_acc'].append(tr['acc']); hist['val_acc'].append(vl['acc'])
            hist['train_bpc'].append(tr['ce']/math.log(2))
            hist['val_bpc'].append(vl['ce']/math.log(2))
            hist['sparsity'].append(vl['sparsity'])
            sp = f" Sp:{vl['sparsity']:.0%}" if not is_gpt else ''
            print(f"  [{label}] {step:5d} | CE:{vl['ce']:.3f} BPC:{vl['ce']/math.log(2):.3f} "
                  f"Acc:{vl['acc']:.1%}{sp}")

        if step >= c.max_steps: break
        st = time.time()
        batch = get_batch(train_data, c)
        opt.zero_grad()
        logits, info = model(batch)
        tgt = batch[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, c.vocab_size), tgt.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        hist['step_times'].append(time.time() - st)
        hist['per_step_loss'].append(loss.item())

    el = time.time() - t0
    ms = np.mean(hist['step_times']) * 1000
    print(f"  {label} DONE: {el/60:.1f}min | BPC:{hist['val_bpc'][-1]:.3f} "
          f"Acc:{hist['val_acc'][-1]:.1%} | {ms:.0f}ms/step")
    hist['avg_step_ms'] = ms; hist['n_params'] = np_
    return hist

## Train All Models

~2-3 hours total at 5000 steps per model. Adjust `cfg.max_steps` above if needed.

In [ ]:
all_hist = {}
for name, model in models.items():
    is_gpt = (name == 'GPT-Nano')
    all_hist[name] = train_model(model, cfg, label=name, is_gpt=is_gpt)

## Results Comparison

In [ ]:
# Colors for each model
colors = {
    'A: Full V14': 'tab:blue',
    'B: No Wilson': 'tab:orange',
    'C: No Langevin': 'tab:green',
    'D: No Sparsity': 'tab:purple',
    'E: SSM+MLP': 'tab:red',
    'GPT-Nano': 'tab:gray',
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('V14 Ablation Study', fontsize=16, fontweight='bold')

# Val BPC
ax = axes[0, 0]
for name, h in all_hist.items():
    ax.plot(h['step'], h['val_bpc'], '-o', color=colors[name], label=name,
            markersize=2, alpha=0.9)
ax.set_xlabel('Step'); ax.set_title('Val BPC (lower = better)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Val Accuracy
ax = axes[0, 1]
for name, h in all_hist.items():
    ax.plot(h['step'], [a*100 for a in h['val_acc']], '-o', color=colors[name],
            label=name, markersize=2, alpha=0.9)
ax.set_xlabel('Step'); ax.set_title('Val Accuracy %')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Smoothed step loss
ax = axes[0, 2]
w = 50
for name, h in all_hist.items():
    if len(h['per_step_loss']) > w:
        sm = np.convolve(h['per_step_loss'], np.ones(w)/w, mode='valid')
        ax.plot(range(len(sm)), sm, '-', color=colors[name], label=name, alpha=0.8)
ax.set_title(f'Step Loss (smooth {w})'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Sparsity (V14 variants only)
ax = axes[1, 0]
for name, h in all_hist.items():
    if name != 'GPT-Nano':
        ax.plot(h['step'], [s*100 for s in h['sparsity']], '-o', color=colors[name],
                label=name, markersize=2, alpha=0.9)
ax.set_xlabel('Step'); ax.set_title('Sparsity (% modes < 0.01)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Step time
ax = axes[1, 1]
step_times = [(name, h['avg_step_ms']) for name, h in all_hist.items()]
step_times.sort(key=lambda x: x[1])
bars = ax.barh([n for n,_ in step_times], [t for _,t in step_times],
               color=[colors[n] for n,_ in step_times])
ax.set_xlabel('ms/step'); ax.set_title('Compute Cost')

# Final results table
ax = axes[1, 2]; ax.axis('off')
rows = []
for name, h in all_hist.items():
    rows.append([name, f"{h['n_params']:,}", f"{h['val_bpc'][-1]:.3f}",
                 f"{h['val_acc'][-1]:.1%}", f"{h['avg_step_ms']:.0f}"])
t = ax.table(cellText=rows,
             colLabels=['Model', 'Params', 'BPC', 'Acc', 'ms/step'],
             loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.8)
ax.set_title('Final Results', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('v14_ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Print verdict
print('\n' + '=' * 70)
print('ABLATION RESULTS')
print('=' * 70)
for name, h in all_hist.items():
    print(f"  {name:<20} BPC:{h['val_bpc'][-1]:.3f}  Acc:{h['val_acc'][-1]:.1%}  "
          f"Params:{h['n_params']:,}  {h['avg_step_ms']:.0f}ms/step")

a_bpc = all_hist['A: Full V14']['val_bpc'][-1]
e_bpc = all_hist['E: SSM+MLP']['val_bpc'][-1]
b_bpc = all_hist['B: No Wilson']['val_bpc'][-1]
c_bpc = all_hist['C: No Langevin']['val_bpc'][-1]
d_bpc = all_hist['D: No Sparsity']['val_bpc'][-1]
tol = 0.03  # within 0.03 BPC = "approximately equal"

print(f"\n--- Verdict ---")
print(f"A vs E (geometry vs SSM+MLP): delta = {a_bpc - e_bpc:+.3f} BPC")
if a_bpc < e_bpc - tol:
    print("  ✓ GEOMETRY EARNS ITS KEEP — Full V14 outperforms SSM+MLP")
elif abs(a_bpc - e_bpc) <= tol:
    print("  ✗ GEOMETRY IS DECORATIVE — same result as V12.2")
else:
    print("  ✗ GEOMETRY HURTS — SSM+MLP is better (overhead without benefit)")

print(f"\nA vs B (Wilson line): delta = {a_bpc - b_bpc:+.3f} BPC")
if a_bpc < b_bpc - tol:
    print("  Wilson line contributes")
else:
    print("  Wilson line is decorative")

print(f"\nA vs C (Langevin): delta = {a_bpc - c_bpc:+.3f} BPC")
if a_bpc < c_bpc - tol:
    print("  Langevin settling contributes")
else:
    print("  Langevin settling is decorative (V14 ≈ complex SSM + linear readout)")

print(f"\nA vs D (Sparsity): delta = {a_bpc - d_bpc:+.3f} BPC")
if a_bpc < d_bpc - tol:
    print("  Proximal sparsity contributes")
else:
    print("  Proximal sparsity doesn't matter yet")

## Mechanism Diagnostics

Per-model analysis of what each geometric component is actually doing.

In [ ]:
@torch.no_grad()
def mechanism_diagnostics(models, cfg):
    print("=" * 70)
    print("MECHANISM DIAGNOSTICS")
    print("=" * 70)

    batch = get_batch(val_data, cfg)

    # --- Model A: Full V14 ---
    model_a = models['A: Full V14']
    model_a.eval()
    c = model_a.embedding(batch)

    print("\n--- Model A: Wilson Line Phase Rotation ---")
    for i, block in enumerate(model_a.blocks):
        normed = block.norm(c)
        flat = normed.to_flat()
        wilson = block.fiber.wilson_proj(flat)
        phase_rot = wilson[..., cfg.n_modes:]
        theta = torch.tanh(phase_rot) * math.pi
        theta_abs = theta.abs().mean().item()
        decay_delta = wilson[..., :cfg.n_modes]
        decay = torch.sigmoid(block.fiber.base_decay + decay_delta)

        messages = block.fiber(normed)
        c = block(c)

        print(f"  Block {i+1}: |θ| = {theta_abs:.4f} rad "
              f"({'ACTIVE' if theta_abs > 0.01 else 'dormant'}) "
              f"| decay = [{decay.min():.2f}, {decay.mean():.2f}, {decay.max():.2f}]")

    # --- Model A: Langevin vs Ctx contribution ---
    print("\n--- Model A: Langevin vs Ctx Skip ---")
    c = model_a.embedding(batch)
    for i, block in enumerate(model_a.blocks):
        normed = block.norm(c)
        messages = block.fiber(normed)

        M = normed.mag.shape[-1]
        x0 = normed.to_flat()
        ctx = block.settler.msg_proj(messages)
        m_norm = F.normalize(block.settler.memory, dim=-1)
        x = x0.clone()
        for k in range(block.settler.K):
            beta = block.settler.beta_min + (block.settler.beta_max - block.settler.beta_min) * k / max(1, block.settler.K - 1)
            q = F.normalize(x + ctx, dim=-1)
            scores = beta * (q @ m_norm.T)
            weights = F.softmax(scores, dim=-1)
            attractor = weights @ block.settler.memory
            grad = attractor - x
            x = x + block.settler.eta * grad

        langevin_norm = (x - x0).norm(dim=-1).mean().item()
        ctx_norm = ctx.norm(dim=-1).mean().item()
        ratio = langevin_norm / max(ctx_norm, 1e-8)

        c = block(c)
        if i in [0, 3, 7]:
            print(f"  Block {i+1}: ‖x-x₀‖ = {langevin_norm:.4f}  "
                  f"‖ctx‖ = {ctx_norm:.4f}  "
                  f"ratio = {ratio:.2f} "
                  f"({'Langevin dominates' if ratio > 2 else 'ctx dominates' if ratio < 0.5 else 'balanced'})")

    # --- Memory bank utilization ---
    print("\n--- Model A: Memory Bank Utilization ---")
    c = model_a.embedding(batch)
    for i, block in enumerate(model_a.blocks):
        normed = block.norm(c)
        messages = block.fiber(normed)
        x = normed.to_flat()
        ctx = block.settler.msg_proj(messages)
        m_norm = F.normalize(block.settler.memory, dim=-1)
        q = F.normalize(x + ctx, dim=-1)
        scores = block.settler.beta_max * (q @ m_norm.T)
        weights = F.softmax(scores, dim=-1)
        entropy = -(weights * (weights + 1e-10).log()).sum(-1).mean().item()
        max_ent = math.log(cfg.n_memory_atoms)
        c = block(c)
        if i in [0, 3, 7]:
            print(f"  Block {i+1}: entropy = {entropy:.2f} / {max_ent:.2f} "
                  f"({entropy/max_ent:.0%} of uniform)")

    # --- Sparsity analysis (Model A vs D) ---
    print("\n--- Sparsity: Model A vs D ---")
    for name in ['A: Full V14', 'D: No Sparsity']:
        m = models[name]
        m.eval()
        c = m.embedding(batch)
        for block in m.blocks:
            c = block(c)
        mag = c.mag
        below = (mag.abs() < cfg.sparsity_threshold).float().mean().item()
        above = (mag.abs() > 2 * cfg.sparsity_threshold).float().mean().item()
        print(f"  {name}: below_thresh={below:.1%} above_2x={above:.1%} "
              f"bimodal={'YES' if below > 0.1 and above > 0.3 else 'NO'}")

    # --- Gradient contribution per component (Model A) ---
    print("\n--- Model A: Gradient Norms ---")
    model_a.train()
    batch2 = get_batch(val_data, cfg)
    logits, _ = model_a(batch2)
    tgt = batch2[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
    loss.backward()

    fiber_g = sum(p.grad.norm().item() for b in model_a.blocks
                  for p in b.fiber.parameters() if p.grad is not None)
    settler_g = sum(p.grad.norm().item() for b in model_a.blocks
                    for p in b.settler.parameters() if p.grad is not None)
    print(f"  Fiber: {fiber_g:.4f}  Settler: {settler_g:.4f}  "
          f"Ratio: {fiber_g/max(settler_g,1e-8):.2f}")
    model_a.zero_grad()
    model_a.eval()

mechanism_diagnostics(models, cfg)

## Interpreting the Results

### If A > E: Geometry earns its keep
The spectral geometric machinery measurably outperforms a standard SSM+MLP.
Check B, C, D to see which mechanism contributes most.

### If A ≈ E: Geometry is still decorative
Same failure mode as V12.2. The constellation representation and Fourier structure
don't add value beyond what a simple EMA+MLP can achieve.

### If A > C ≈ E: Complex Mamba
The Wilson fiber (content-dependent complex SSM) is the key contributor.
The Langevin settling is decorative. V14 is effectively "complex Mamba"
with Parseval inner products. Still a valid architecture — just not the
full theory.

### If A > B ≈ E: Wilson line is the mechanism
Content-dependent phase rotation is what breaks the plateau.
The gauge connection / holonomy interpretation is validated.

### What the diagnostics reveal
- **|θ| > 0.01**: Wilson line is active (learning non-trivial phase rotations)
- **‖x-x₀‖/‖ctx‖ > 0.5**: Langevin is doing real work (not just a gradient highway)
- **Bimodal magnitudes**: Proximal sparsity is enforcing spectral parsimony
- **Memory entropy < max**: Hopfield is using specific attractors, not uniform
- **Fiber/Settler gradient ratio > 0.1**: Both components are learning

## Findings (Tiny Shakespeare, 3500 steps, 2026-03-31)

### Results

| Model | BPC | Acc | Params | ms/step |
|---|---|---|---|---|
| GPT-Nano | **2.229** | **55.2%** | 2.46M | 65 |
| **D: No Sparsity** | **2.395** | **52.9%** | 1.80M | 298 |
| A: Full V14 | 2.507 | 50.1% | 1.80M | 311 |
| B: No Wilson | 2.541 | 50.6% | 0.96M | 140 |
| C: No Langevin | 2.543 | 51.1% | 1.24M | 161 |
| E: SSM+MLP | 2.772 | 44.4% | 1.80M | 107 |

### Verdicts

**A >> E (+0.265 BPC): Geometry earns its keep.**
This is the opposite of V12.2. The geometric constellation + fiber + memory bank
provides substantial, measurable value over SSM+MLP at the same parameter count.
The spectral machinery is no longer decorative.

**D > A (+0.112 BPC): Sparsity actively hurts.**
Removing the proximal threshold makes the model BETTER. At vocab 65, there is no
compression pressure. The threshold destroys useful magnitude information. Drop it.

**B ≈ C ≈ A: Wilson line and Langevin are substitutes, not complements.**
Each alone captures ~0.23 BPC improvement over E. Combining them doesn't stack.
The Hopfield memory bank (present in both B and A) is the key contributor, providing
content-addressable retrieval that the MLP cannot replicate.

**B is the efficiency winner**: 2.541 BPC with half the parameters (957K) and
half the compute (140ms) of full V14. Fixed-decay fiber + Langevin memory bank
is the highest bang-per-parameter configuration.

### Key Takeaways

1. **The Hopfield memory bank is the real contributor**, not the Wilson line phase rotation.
   Model B (fixed decay + memory bank) matches A (Wilson + memory bank). Content-dependent
   phase rotation adds nothing measurable on this task.

2. **Sparsity is a net negative** at vocab 65. No compression pressure exists. Drop it immediately.

3. **The winning recipe for hybrid architectures**: fiber + Hopfield memory bank + MLP.
   This combines B's retrieval mechanism with V12.1's per-token expressivity.

4. **The Wilson line may still matter at scale** — character-level Shakespeare doesn't test
   long-range contextual curvature. BPE + larger vocab + longer contexts are needed to
   properly evaluate content-dependent transport.

5. **D (No Sparsity) is still improving** at 3500 steps (2.443 → 2.395 over last 750 steps)
   while GPT-Nano has plateaued at 2.229. The gap may close further with more training.

### Caveats

- B and C have fewer parameters than A (957K and 1.24M vs 1.80M). The parameter
  difference partially confounds the comparison. However, B's smaller model performing
  comparably to A's larger model strengthens the case that the Wilson line's extra
  parameters aren't earning their keep.
- E (SSM+MLP) uses a scalar gate (like V13) rather than per-mode gates. This may
  partially explain its weaker performance vs the geometric variants.
- 3500 steps may not be enough for full convergence. D in particular was still improving.
- These findings are specific to character-level Tiny Shakespeare (vocab 65, seq 512, 1.1M chars).
  They should not be extrapolated to larger-vocab, longer-context regimes without verification.